In [ ]:
from mistralai import Mistral

In [6]:
# Sample Run from Mistral Docs

import pandas as pd

DEFAULT_MODEL: "mistral-large-latest"
SOLVER_MODEL:  "mistral-large-latest"
EVOLVE_MODEL: "mistral-large-latest"
NON_SOLVER_MODELS: "mistral-large-latest"

# # Assuming we have the following data
# data = {
#     'transaction_id': ['T1001', 'T1002', 'T1003', 'T1004', 'T1005'],
#     'customer_id': ['C001', 'C002', 'C003', 'C002', 'C001'],
#     'payment_amount': [125.50, 89.99, 120.00, 54.30, 210.20],
#     'payment_date': ['2021-10-05', '2021-10-06', '2021-10-07', '2021-10-05', '2021-10-08'],
#     'payment_status': ['Paid', 'Unpaid', 'Paid', 'Paid', 'Pending']
# }

# # Create DataFrame
# df = pd.DataFrame(data)

# def retrieve_payment_status(df: data, transaction_id: str) -> str:
#     if transaction_id in df.transaction_id.values: 
#         return json.dumps({'status': df[df.transaction_id == transaction_id].payment_status.item()})
#     return json.dumps({'error': 'transaction id not found.'})

# def retrieve_payment_date(df: data, transaction_id: str) -> str:
#     if transaction_id in df.transaction_id.values: 
#         return json.dumps({'date': df[df.transaction_id == transaction_id].payment_date.item()})
#     return json.dumps({'error': 'transaction id not found.'})

tools = [
    {
        "type": "function",
        "function": {
            "name": "action_display_analysis",
            "description": "Display an analysis of the current state, including available resources, logic (of solver or other actions) and evaluation feedbacks from the target task, and reasons or plans for the next actions based on this analysis.",
            "parameters": {
                "type": "object",
                "properties": {
                    "analysis": {
                        "type": "string",
                        "description": "A detailed analysis of the current state, including reasons or plans for the following actions."
                    }
                },
                "required": ["analysis"],
                "additionalProperties": False,
            },
            "strict": True
        }
    },
    {
        "type": "function",
        "function": {
            "name": "action_environment_aware",
            "description": "Reflect and summarize available resources of the current runtime environment including variables, functions, modules, and external libraries.",
            "parameters": {
                "type": "object",
                "properties": {},
                "required": [],
                "additionalProperties": False
            },
            "strict": True
        }
    },
    {
        "type": "function",
        "function": {
            "name": "action_read_logic",
            "description": "Reads the source code of the specified logic (function, method, or class) within a given module.",
            "parameters": {
                "type": "object",
                "properties": {
                    "module_name": {
                        "type": "string",
                        "description": "The module where the logic resides."
                    },
                    "target_name": {
                        "type": "string",
                        "description": "The name of the function, method, or class to read. If the target_name contains a dot, it refers to a method within a class (e.g., 'Agent.action_call_llm')."
                    }
                },
                "required": ["module_name", "target_name"],
                "additionalProperties": False
            },
            "strict": True
        }
    },
    {
        "type": "function",
        "function": {
            "name": "action_adjust_logic",
            "description": "Modify/Add/Delete the source code of the specified logic (function, method, or class) within a given module to improve task-solving ability or create a tool designed specifically to assist in task-solving efficiently.",
            "parameters": {
                "type": "object",
                "properties": {
                    "module_name": {
                        "type": "string",
                        "description": "The module where the logic resides."
                    },
                    "target_name": {
                        "type": "string",
                        "description": "The name of the function, method, or class to modify/add/delete. If the target_name contains a dot, it refers to a method within a class."
                    },
                    "new_code": {
                        "type": "string",
                        "description": "The new logic as a string. (Ensure there is no extra indentation in new_code)"
                    },
                    "target_type": {
                        "type": "string",
                        "enum": ["function", "class"],
                        "description": "The type of target."
                    },
                    "operation": {
                        "type": "string",
                        "enum": ["modify", "add", "delete"],
                        "description": "The operation to perform."
                    }
                },
                "required": ["module_name", "target_name", "new_code", "target_type", "operation"],
                "additionalProperties": False
            },
            "strict": True
        }
    },
    {
        "type": "function",
        "function": {
            "name": "action_run_code",
            "description": "Execute Python or shell code and capture the output, errors, and return value. (Running python code can get and store objects designed specifically to assist in task-solving efficiently, such as prompts)",
            "parameters": {
                "type": "object",
                "properties": {
                    "code_type": {
                        "type": "string",
                        "enum": ["python", "bash"],
                        "description": "The type of code to execute."
                    },
                    "code": {
                        "type": "string",
                        "description": "The code to execute as a string."
                    }
                },
                "required": ["code_type", "code"],
                "additionalProperties": False
            },
            "strict": True
        }
    },
    {
        "type": "function",
        "function": {
            "name": "action_call_json_format_llm",
            "description": "Call an external LLM for assistance with gathering insights, refining strategies, correcting errors, and solving complex problems. Output response in JSON format.",
            "parameters": {
                "type": "object",
                "properties": {
                    "model": {
                        "type": "string",
                        "description": "ID of the model to use."
                    },
                    "messages": {
                        "type": "array",
                        "items": {
                            "type": "object",
                            "properties": {
                                "role": {
                                    "enum": ["system", "assistant", "user"]
                                },
                                "content": {"type": "string"}
                            },
                            "required": ["role", "content"],
                            "additionalProperties": False
                        },
                        "description": "A list of messages comprising the conversation so far."
                    },
                    "temperature": {
                        "type": "number",
                        "description": "What sampling temperature to use. Higher values will make the output more random, while lower values will make it more focused and deterministic."
                    },
                    "role": {
                        "type": "string",
                        "description": "The role that LLM play."
                    },
                    "return_dict_keys": {
                        "type": "array",
                        "items": {
                            "type": "string"
                        },
                        "description": "An array containing the names of the keys that should be present in the returned dictionary."
                    },
                    "requirements": {
                        "type": "string",
                        "description": "A string that specifies the conditions required to perform a call to the LLM."
                    }
                },
                "required": ["model", "messages", "temperature", "role", "return_dict_keys", "requirements"],
                "additionalProperties": False
            },
            "strict": True
        }
    },
    {
        "type": "function",
        "function": {
            "name": "action_evaluate_on_task",
            "description": "Evaluate the current solver on the goal task samples and return the evaluation feedback including valid set accuracy, test set accuray, test sample inputs, model outputs and valid sample answer.",
            "parameters": {
                "type": "object",
                "properties": {},
                "required": [],
                "additionalProperties": False
            },
            "strict": True
        }
    }
]

import functools

# names_to_functions = {
#     'retrieve_payment_status': functools.partial(retrieve_payment_status, df=df),
#     'retrieve_payment_date': functools.partial(retrieve_payment_date, df=df)
# }

messages = [
    {
      "role": "system",
      "name": "Principles",
      "content": "You are a **self-evolving agent**, named `self_evolving_agent`, an instance of the `Agent` class, in module `agent_module`, running within an active **Python runtime environment**. You have full access to global variables, functions and modules. Your primary goal is to continuously enhance your ability to solve tasks accurately and efficiently by dynamically reflecting environment and evolving your logic.\n\n### **Core Capabilities**:\n\n+ **Complete Autonomy**: Have **unrestricted access** to modify logic, run code and manipulate environment.\n+ **Environment Interaction**: Interact with the environment by perceiving environment, reading or modifying or executing code and executing actions.\n+ **Problem-Solving**: Apply creative algorithms or self-developed structures to tackle challenges when simple methods fall short, optimizing solutions effectively.\n+ **Collaboration**: Leverage OpenAI LLM to gather insights, refine strategies, correct errors, and solve complex problems.\n+ **Error Handling**: Carefully analyze errors. When errors occur, troubleshoot systematically, and if a bug is persistent, backtrack, restore the original state, or find an alternative solution.\n\n### **Core Methods**:\n\n+ **`evolve`**: Continuously enhance performance by interacting with environment.\n+ **`execute_action(actions)`**: Execute actions based on analysis or feedback.\n+ **`solver(agent_instance, task_input: str)`**: Solve the target task using current `agent_instance` capabilities, and objects created by `action_adjust_logic` and `action_run_code`, optimizing the process.\n\n### **Guiding Principles**:\n\n+ **Remember** that all functions are in module `agent_module`. \n+ **Remember** that you can call **MULTIPLE tools** at once.\n+ **`action_adjust_logic`**: \n    + Before modifying the code, make sure that each variable or function used is used and imported correctly to avoid errors. \n    + Do not do unnecessary changes. \n    + Do not change interface of any function. \n    + Can be use to create action functions for `solver`.\n+ **`action_run_code`**: \n    + Make sure that each variable or function used is used and imported correctly to avoid errors. \n    + ALL created objects in Python mode can be stored in environment.\n    + Can be use to create objects for `solver`, such as prompt. \n    + Can be use to import new module or external libraries and install external libraries.\n+ **External Collaboration**: Seek external assistance via `action_call_json_format_llm` for logic refinement and new tool creation or `action_run_code` to execute code and then get and store the useful objects, like PROMPTS, that can be reused in `solver`.\n+ **`action_evaluate_on_task`**: Assess the performance of `solver` ONLY after successfully modifying the logic of `solver`.\n+ **`solver`**:\n    + Is defined as `agent_module.solver`.\n    + The output MUST be a dictionary, and the final answer MUST be placed under the key `\"answer\"`.\n    + For debugging, don't print, and instead return the debug information.\n    + When calling OpenAI LLMs, it must exclusively use `action_call_json_format_llm`, and only the **mistral-large-latest** model is allowed for such calls.\n    + Can call `action_call_json_format_llm` multiple times and across multiple rounds in the solver to improve performance.\n    + If performance doesn't improve, explore alternative methods.\n    + When multiple outputs are required, set `num_of_response`, a parameter of `action_call_json_format_llm`, to the required number of outputs in the function.\n    + Additionally, can call different role-based LLMs by specifying and MUST specifying the role to further assist task-solving.\n    + For each key, if a specific format is required, such as int, float, enum or list, the requirements must specify the conditions.\n    + Explore techniques like:\n        + **Large Language Model Debate**: Multiple models engage in a discussion to critique and refine responses, improving solution quality.\n        + **Step-back Abstraction**: Solving problems by shifting to a higher, more abstract perspective to simplify and break down complex tasks.\n        + **Quality-Diversity**: Focusing on generating diverse, high-quality solutions rather than exclusively optimizing one outcome.\n        + **Dynamic Assignment of Roles**: Assigning and adjusting roles among AI components dynamically to enhance task performance.\n        + **Self-consistency**: Ensure coherence by comparing multiple outputs and selecting the most consistent one. (Can try to increase `num_of_response` to get high score)\n        + **Few-shots**: Using few-shot learning to quickly adapt with minimal examples(can use valid examples), improving performance on new tasks through generalization.\n        + **Task Decomposition**: Dividing complex tasks into smaller subtasks, solving them individually, and reintegrating the solutions for overall task success.\n        + **Reflective Evaluation**: Reviewing performance after task completion to identify successes and failures, enabling continuous self-improvement.\n    + Can combine above techniques.\n+ **`action_display_analysis`**: \n    + **Always analysis first before acting.** \n    + Analysis may include following things: reasonable plan about improving performance, **CASE STUDIES of LOW SCORE valid examples of EVALUATION FEEDBACK**, error handling, other possible solving ideas. \n    + **If performance does not improve, conduct further analysis.**\n    + `action_call_json_format_llm` can also do analysis."
    },
    {
      "role": "system",
      "name": "Environment",
      "content": "Global Functions:\n- get_openai_instance = <function get_openai_instance at 0x7fbe243074c0>\n- log_model_input_output = <function log_model_input_output at 0x7fbd84c905e0>\n- action_display_analysis = <function action_display_analysis at 0x7fbe3de920d0>\n- action_environment_aware = <function action_environment_aware at 0x7fbd84c90790>\n- action_read_logic = <function action_read_logic at 0x7fbd84c90820>\n- action_adjust_logic = <function action_adjust_logic at 0x7fbd84c908b0>\n- action_run_code = <function action_run_code at 0x7fbd84c90940>\n- solver = <function solver at 0x7fbd84c909d0>\n- action_evaluate_on_task = <function action_evaluate_on_task at 0x7fbd84b70310>\n\nGlobal Modules:\n- os = <module 'os' from '/local/home/nbelle/.conda/envs/godel/lib/python3.9/os.py'>\n- io = <module 'io' from '/local/home/nbelle/.conda/envs/godel/lib/python3.9/io.py'>\n- re = <module 're' from '/local/home/nbelle/.conda/envs/godel/lib/python3.9/re.py'>\n- ast = <module 'ast' from '/local/home/nbelle/.conda/envs/godel/lib/python3.9/ast.py'>\n- sys = <module 'sys' (built-in)>\n- json = <module 'json' from '/local/home/nbelle/.conda/envs/godel/lib/python3.9/json/__init__.py'>\n- typing = <module 'typing' from '/local/home/nbelle/.conda/envs/godel/lib/python3.9/typing.py'>\n- inspect = <module 'inspect' from '/local/home/nbelle/.conda/envs/godel/lib/python3.9/inspect.py'>\n- functools = <module 'functools' from '/local/home/nbelle/.conda/envs/godel/lib/python3.9/functools.py'>\n- itertools = <module 'itertools' (built-in)>\n- traceback = <module 'traceback' from '/local/home/nbelle/.conda/envs/godel/lib/python3.9/traceback.py'>\n- importlib = <module 'importlib' from '/local/home/nbelle/.conda/envs/godel/lib/python3.9/importlib/__init__.py'>\n- subprocess = <module 'subprocess' from '/local/home/nbelle/.conda/envs/godel/lib/python3.9/subprocess.py'>\n- contextlib = <module 'contextlib' from '/local/home/nbelle/.conda/envs/godel/lib/python3.9/contextlib.py'>\n- collections = <module 'collections' from '/local/home/nbelle/.conda/envs/godel/lib/python3.9/collections/__init__.py'>\n- openai = <module 'openai' from '/local/home/nbelle/.conda/envs/godel/lib/python3.9/site-packages/openai/__init__.py'>\n- task_mgsm = <module 'task_mgsm' from '/share/edc/home/mbelle/projects/godel/Godel_Agent/src/task_mgsm.py'>\n\nGlobal Variables:\n- ModelType = typing.Literal['mistral-large-latest']\n- DEFAULT_MODEL = mistral-large-latest\n- SOLVER_MODEL = mistral-large-latest\n- NON_SOLVER_MODELS = []\n- EVOLVE_MODEL = mistral-large-latest\n- action_counter = defaultdict(<class 'int'>, {'evolve': 1})\n- self_evolving_agent = <agent_module.Agent object at 0x7fbe3ce583d0>\n\nGlobal Classes:\n- Mistral = <class 'mistralai.sdk.Mistral'>\n- Agent = <class 'agent_module.Agent'>\n\nCurrent Agent Instance's Methods:\n- action_call_json_format_llm = <bound method Agent.action_call_json_format_llm of <agent_module.Agent object at 0x7fbe3de8c880>>\n- action_call_llm = <bound method Agent.action_call_llm of <agent_module.Agent object at 0x7fbe3de8c880>>\n- evolve = <bound method Agent.evolve of <agent_module.Agent object at 0x7fbe3de8c880>>\n- execute_action = <bound method Agent.execute_action of <agent_module.Agent object at 0x7fbe3de8c880>>\n- reinit = <bound method Agent.reinit of <agent_module.Agent object at 0x7fbe3de8c880>>\n\nCurrent Agent Instance's Attributes:\n- action_functions = The length of your action_functions is 7.\n- client = <mistralai.sdk.Mistral object at 0x7fbe3de8cfd0>\n- goal_prompt = Your goal_prompt.\n- goal_task = <task_mgsm.MGSM_Task object at 0x7fbe3de8cfa0>\n- optimize_history = The length of your optimize_history is 1."
    },
    {
      "role": "user",
      "content": "The logic of solver:\ndef solver(agent, task: str):\n    messages = [{\"role\": \"user\", \"content\": f\"# Your Task:\\n{task}\"}]\n    response = agent.action_call_json_format_llm(\n        model=SOLVER_MODEL,\n        messages=messages, \n        temperature=0.8, \n        num_of_response=1,\n        role=\"math expert\", \n        return_dict_keys=[\"reasoning\", \"answer\"], \n        requirements=(\n            \"1. Please explain step by step.\\n\"\n            \"2. The answer MUST be an integer.\\n\"\n        ).strip(),\n    )\n    \n    return_dict = response[0]\n    return_dict[\"answer\"] = str(return_dict.get(\"answer\", \"\"))\n    return return_dict\n"
    }
  ]

import os
from mistralai import Mistral

api_key = "z8ub8Rgj6Mezu87vCm2qSzq2uGScJRDS"
model = "mistral-large-latest"

client = Mistral(api_key=api_key)
response = client.chat.complete(
    model = model,
    messages = messages,
    tools = tools,
    tool_choice = "any",
)
print(response)

id='28d666600c7b45b8bebb5ff3568723f5' object='chat.completion' model='mistral-large-latest' usage=UsageInfo(prompt_tokens=4208, completion_tokens=272, total_tokens=4480) created=1738723999 choices=[ChatCompletionChoice(index=0, message=AssistantMessage(content='', tool_calls=[ToolCall(function=FunctionCall(name='action_display_analysis', arguments='{"analysis": "The solver currently utilizes the LLM model \'mistral-large-latest\' to solve the task by providing the task content. It sets the temperature to 0.8 and requests one response.\\n\\nTo improve the solver\'s performance, we can consider the following approaches:\\n\\n1. **Large Language Model Debate**: Engage multiple models in a discussion to refine and critique responses, enhancing solution quality.\\n2. **Self-Consistency**: Increase the number of responses (e.g., num_of_response=3) and select the most consistent one as the final answer.\\n3. **Few-Shot Learning**: Incorporate valid samples into the prompt to help the model qu

In [2]:
# Tool definitions
tools = [
    {
        "type": "function",
        "function": {
            "name": "query_daily_sales_report",
            "description": "Connects to a database to retrieve overall sales volumes and sales information for a given day.",
            "parameters": {
                "type": "object",
                "properties": {
                    "day": {
                        "type": "string",
                        "description": "Retrieves sales data for this day, formatted as YYYY-MM-DD.",
                    }
                },
                "required": ["day"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "query_product_catalog",
            "description": "Connects to a product catalog with information about all the products being sold, including categories, prices, and stock levels.",
            "parameters": {
                "type": "object",
                "properties": {
                    "category": {
                        "type": "string",
                        "description": "Retrieves product information data for all products in this category.",
                    }
                },
                "required": ["category"],
            },
        },
    },
]


In [4]:
system_message = """
## Task & Context
You help people answer their questions and other requests interactively. You will be asked a very wide array of requests on all kinds of topics. You will be equipped with a wide range of search engines or similar tools to help you, which you use to research your answer. You should focus on serving the user's needs as best you can, which will be wide-ranging.

## Style Guide
Unless the user asks for a different style of answer, you should answer in full sentences, using proper grammar and spelling.
"""

# user request
message = "Can you provide a sales summary for 29th September 2023, and also give me some details about the products in the 'Electronics' category, for example their prices and stock levels?"

messages = [
    {"role": "system", "content": system_message},
    {"role": "user", "content": message},
]


In [5]:
response = co.chat(
    model="command-r-plus-08-2024", messages=messages, tools=tools
)

print("The model recommends doing the following tool calls:\n")
print("Tool plan:")
print(response.message.tool_plan, "\n")
print("Tool calls:")
for tc in response.message.tool_calls:
    print(
        f"Tool name: {tc.function.name} | Parameters: {tc.function.arguments}"
    )

# append the chat history
messages.append(
    {
        "role": "assistant",
        "tool_calls": response.message.tool_calls,
        "tool_plan": response.message.tool_plan,
    }
)


The model recommends doing the following tool calls:

Tool plan:
I will first search for the sales summary for 29th September 2023. Then, I will search for the products in the 'Electronics' category. 

Tool calls:
Tool name: query_daily_sales_report | Parameters: {"day":"2023-09-29"}
Tool name: query_product_catalog | Parameters: {"category":"Electronics"}
